# Kaggle Training Orchestration — amharic-efficient-summarization

Thin orchestration notebook: it does **not** contain training logic itself.
It clones the repo, installs dependencies, runs the data/subset-building
pipeline, then loops over all 9 runs in the experiment matrix, calling
`src/train.py` and `src/evaluate.py` as subprocesses for each run.

Runs on a Kaggle T4 GPU. Local dev happens on WSL Ubuntu (no local GPU).

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone repo and install dependencies

In [ ]:
REPO_URL = "https://github.com/yakobd/gheero-amharic-efficient-summarization.git"  # placeholder
REPO_DIR = "amharic-efficient-summarization"

!git clone $REPO_URL $REPO_DIR

In [ ]:
%cd {REPO_DIR}
!pip install -r requirements.txt

## 3. Data pipeline: prep -> quality scoring -> diversity clustering -> build subsets

In [ ]:
!python src/data_prep.py
!python src/quality_scoring.py
!python src/diversity_clustering.py
!python src/build_subsets.py

## 4. Run the 9-run experiment matrix

Each entry is `(run_name, subset_path)`. `full_dataset` trains on the full
processed training split; the other 8 runs train on the corresponding
subset produced by `src/build_subsets.py` in `data/subsets/`.

In [ ]:
import subprocess

RUNS = [
    ("full_dataset", "data/processed/train.jsonl"),
    ("random_25pct", "data/subsets/random_25pct.jsonl"),
    ("quality_only_25pct", "data/subsets/quality_only_25pct.jsonl"),
    ("diversity_only_25pct", "data/subsets/diversity_only_25pct.jsonl"),
    ("combined_25pct", "data/subsets/combined_25pct.jsonl"),
    ("random_50pct", "data/subsets/random_50pct.jsonl"),
    ("quality_only_50pct", "data/subsets/quality_only_50pct.jsonl"),
    ("diversity_only_50pct", "data/subsets/diversity_only_50pct.jsonl"),
    ("combined_50pct", "data/subsets/combined_50pct.jsonl"),
]

In [ ]:
for run_name, subset_path in RUNS:
    print(f"=== TRAIN: {run_name} ===")
    checkpoint_dir = f"results/checkpoints/{run_name}"
    subprocess.run(
        ["python", "src/train.py", "--run_name", run_name, "--subset_path", subset_path],
        check=True,
    )

    print(f"=== EVALUATE: {run_name} ===")
    subprocess.run(
        ["python", "src/evaluate.py", "--run_name", run_name, "--checkpoint_dir", checkpoint_dir],
        check=True,
    )

## 5. Zip results for download

In [ ]:
!zip -r results.zip results/